# CP201A Lab 5 Companion: Crosswalking 2019 Tract Data onto 2020 Tracts

**Fall 2026. Advanced path only.**

Assignment 1 Part II compares 2015 to 2019 with 2020 to 2024 at the neighborhood scale for
students on the advanced path. The 2019 estimates sit on 2010 census tracts and the 2024
estimates on 2020 tracts, and the Census Bureau redrew some tracts in 2020. This notebook
handles the cases the direct pull in Lab 5 Section 6.2 cannot: a tract that was merged,
renumbered, or had its boundary moved.

## When you need this

Check your tract list in the starter tract file or the Tract Codebook notebook (both on
bCourses), then:

* **Every tract unchanged:** you do not need this. Pull 2019 with the same list (Lab 5, 6.2).
* **A tract was split in 2020:** you do not need this either. Give the 2019 pull the single
  2010 number and the 2024 pull its pieces; they cover the same land, so the neighborhood
  totals line up.
* **A tract was merged, renumbered, or its boundary moved:** the 2010 and 2020 lists do not
  cover the same land, and the direct pull silently drops or misplaces people. This notebook
  reallocates the 2019 numbers onto the 2020 tracts you are using.

The example is three 2020 tracts in Berkeley chosen because all three changed: tract 9821
(the campus, which was tract 4226 in 2010) and tracts 4229.01 and 4229.02 (a 2020 split of
tract 4229). A direct 2019 pull of those three numbers returns nothing at all.

## What a crosswalk is

A crosswalk is a table with one row for each piece of land that a 2010 tract and a 2020
tract have in common, with the area of that piece. The Census Bureau publishes one for every
state (the 2020 Census tract relationship files). From it we compute, for each 2010 tract,
what share of its land ended up in each 2020 tract, and we hand that share of its people
over. A 2010 tract entirely inside one of your 2020 tracts contributes all of itself; one
that is 30 percent inside contributes 30 percent.

The assumption is that people are spread evenly across a tract's land. They are not, and in
places with parks, water, freeways, or a port the assumption can be far off. It is the
standard simple method, it is what a relationship file supports, and it is a great deal
better than pretending the boundaries did not move. Section 6 names the better tools and
what they cost.

## 1. Setup

The same start as Lab 5, plus the four Lab 4 functions and the B03002 dictionary. Change the
dictionary to the table your Part II question uses; the crosswalk step is the same for any
**count** table. (Medians and means do not crosswalk; see Section 6.)

In [ ]:
%pip install -q census

In [ ]:
from census import Census
import pandas as pd
import numpy as np
import os

try:
    with open(os.path.expanduser('~/census_key.txt')) as f:
        api_key = f.read().strip()
    print('Key loaded. It starts with:', api_key[:4] + '...')
except FileNotFoundError:
    print('No key file found. Open Lab 3, run the cell in Section 0.1 once to save your key, then run this cell again.')

c = Census(key=api_key)

In [ ]:
# ---- Copied from Lab 4 and Lab 5. Nothing new here. ----
variables_of_interest = {
    'NAME': 'NAME', 'GEO_ID': 'GEO_ID',
    'B03002_001E': 'total', 'B03002_001M': 'total_moe',
    'B03002_003E': 'nh_white', 'B03002_003M': 'nh_white_moe',
    'B03002_004E': 'nh_black', 'B03002_004M': 'nh_black_moe',
    'B03002_005E': 'nh_native', 'B03002_005M': 'nh_native_moe',
    'B03002_006E': 'nh_asian', 'B03002_006M': 'nh_asian_moe',
    'B03002_007E': 'nh_pi', 'B03002_007M': 'nh_pi_moe',
    'B03002_008E': 'nh_1other', 'B03002_008M': 'nh_1other_moe',
    'B03002_009E': 'nh_multi', 'B03002_009M': 'nh_multi_moe',
    'B03002_012E': 'hispanic', 'B03002_012M': 'hispanic_moe',
}
GROUPS = ['hispanic', 'nh_white', 'nh_black', 'nh_native', 'nh_asian', 'nh_pi', 'nh_1other', 'nh_multi']


def clean_acs(df, id_cols=('NAME', 'GEO_ID', 'state', 'county', 'tract', 'place')):
    df = df.copy()
    numeric_cols = [col for col in df.columns if col not in id_cols]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col])
    moe_cols = [col for col in numeric_cols if col.endswith('_moe')]
    df[moe_cols] = df[moe_cols].replace(-555555555, 0)
    df[numeric_cols] = df[numeric_cols].replace([-666666666, -222222222, -333333333], np.nan)
    return df


def aggregate_tracts(df, name):
    numeric_cols = df.select_dtypes('number').columns
    moe_cols = [col for col in numeric_cols if col.endswith('_moe')]
    est_cols = [col for col in numeric_cols if col not in moe_cols]
    estimates = df[est_cols].sum()
    moes = (df[moe_cols]**2).sum()**0.5
    out = pd.DataFrame(pd.concat([estimates, moes])).transpose()
    out.insert(0, 'NAME', name)
    out.insert(1, 'n_tracts', len(df))
    return out


def add_shares(df, groups, total='total'):
    df = df.copy()
    y = df[total]
    moe_y = df[f'{total}_moe']
    for g in groups:
        p = df[g] / y
        under_root = df[f'{g}_moe']**2 - p**2 * moe_y**2
        under_root = np.where(under_root < 0, df[f'{g}_moe']**2 + p**2 * moe_y**2, under_root)
        df[f'pct_{g}'] = p * 100
        df[f'pct_{g}_moe'] = under_root**0.5 / y * 100
    return df


def pull_geo(geo_for, geo_in, variables=variables_of_interest, year=2024):
    df = pd.DataFrame(
        c.acs5.get(list(variables.keys()), {'for': geo_for, 'in': geo_in}, year=year)
    ).rename(columns=variables)
    return clean_acs(df)


def add_standard_errors(df):
    df = df.copy()
    for col in df.columns:
        if col.endswith('_moe'):
            df[col[:-4] + '_se'] = df[col] / 1.645
    return df


def z_statistic(df, col, place_1, place_2):
    x1, x2 = df.loc[place_1, col], df.loc[place_2, col]
    se1, se2 = df.loc[place_1, col + '_se'], df.loc[place_2, col + '_se']
    return abs(x1 - x2) / (se1**2 + se2**2)**0.5


def significance(z):
    if z > 2.576:
        return '99 percent'
    elif z > 1.960:
        return '95 percent'
    elif z > 1.645:
        return '90 percent'
    return 'not significant at 90 percent'

print('Toolkit loaded.')

## 2. Your parameters

`TRACT_LIST` holds the **2020** tract numbers you use for 2020 to 2024, exactly as in Lab 4.
The crosswalk finds the 2010 tracts for you; you do not type a 2010 list here.

In [ ]:
NEIGHBORHOOD_NAME = 'Campus and Southside example'
TRACT_LIST = ['982100', '422901', '422902']     # 2020 tract numbers (six digits)
STATE = '06'
COUNTY = '001'
RELATIONSHIP_FILE = 'tab20_tract20_tract10_st06.txt'     # California; see Section 3 for another state

## 3. The relationship file

The file `tab20_tract20_tract10_st06.txt` is the Census Bureau's 2020-to-2010 tract
relationship file for California, downloaded from
https://www.census.gov/geographies/reference-files/time-series/geo/relationship-files.2020.html
and saved in this folder. Columns are separated by `|`. The columns we need:

* `GEOID_TRACT_20` and `GEOID_TRACT_10`: the full 11-digit codes (state + county + tract) of
  the 2020 and 2010 tracts that overlap
* `AREALAND_TRACT_10`: the land area of the whole 2010 tract, in square meters
* `AREALAND_PART`: the land area of the overlapping piece

The weight we want is `AREALAND_PART / AREALAND_TRACT_10`: the share of the 2010 tract's land
that lies inside this 2020 tract. We read the codes as text (`dtype=str`) so that pandas keeps
the leading zero in California's state code.

**A neighborhood in another state:** the Bureau publishes one file per state, named
`tab20_tract20_tract10_stXX.txt` with XX the state's two-digit FIPS code, on the page linked
above (under Census Tract Relationship Files). Download your state's file, upload it to this
folder with the Datahub upload button, and set `RELATIONSHIP_FILE` in Section 2 to its name.
Set `STATE` and `COUNTY` to match. Nothing else changes.

In [ ]:
crosswalk = pd.read_csv(
    RELATIONSHIP_FILE,
    sep='|',
    dtype={'GEOID_TRACT_20': str, 'GEOID_TRACT_10': str},
    encoding='utf-8-sig',          # the file starts with a byte-order mark; this drops it
)
crosswalk['w10'] = crosswalk['AREALAND_PART'] / crosswalk['AREALAND_TRACT_10']
print(f'{len(crosswalk):,} overlap pieces in California.')
crosswalk[['GEOID_TRACT_20', 'NAMELSAD_TRACT_20', 'GEOID_TRACT_10', 'NAMELSAD_TRACT_10', 'AREALAND_PART', 'w10']].head()

## 4. Which 2010 tracts feed your 2020 tracts?

Build the 11-digit codes for your tracts, keep the crosswalk rows for them, and look. Each row
is one piece: a 2010 tract and how much of it is inside one of your 2020 tracts. Tiny weights
(a fraction of a percent) are slivers from the Bureau redrawing a line along a street or a
shoreline; they add almost nothing and do no harm, so we keep them.

In [ ]:
my_geoids_20 = [STATE + COUNTY + t for t in TRACT_LIST]

pieces = crosswalk[crosswalk['GEOID_TRACT_20'].isin(my_geoids_20)].copy()
pieces = pieces[pieces['AREALAND_PART'] > 0]

# Any 2010 source in another county? It would need that county pulled too (Section 5 handles it).
pieces['county_10'] = pieces['GEOID_TRACT_10'].str[:5]
other_county = pieces[(pieces['county_10'] != STATE + COUNTY) & (pieces['w10'] > 0.01)]
if len(other_county):
    print('2010 tracts in another county contribute more than 1 percent of their land:')
    print(other_county[['GEOID_TRACT_20', 'GEOID_TRACT_10', 'w10']].to_string(index=False))

pieces[['NAMELSAD_TRACT_20', 'GEOID_TRACT_10', 'NAMELSAD_TRACT_10', 'AREALAND_PART', 'w10']].round(4)

Read the example: 2020 tract 9821 is 98.6 percent of 2010 tract 4226 plus slivers of four
neighbors; tracts 4229.01 and 4229.02 are 29 and 71 percent of 2010 tract 4229.

Now one step that matters for the margins of error. We are building a **neighborhood** total,
not a value for each 2020 tract, so what we need from each 2010 tract is its **total** weight
across all of our 2020 tracts. Tract 4229 appears twice above (once per piece); together its
pieces are 100 percent of it, so the whole tract should go in, with its whole margin of error.
If we allocated the two pieces separately and then combined them by root sum of squares, we
would be treating two halves of the same estimate as if they were independent, and the MOE
would come out too small. So: group by 2010 tract and sum the weights first.

In [ ]:
weights = pieces.groupby('GEOID_TRACT_10', as_index=False)['w10'].sum()
weights = weights.rename(columns={'w10': 'weight'})
weights['county_10'] = weights['GEOID_TRACT_10'].str[:5]
weights.round(4)

## 5. Pull 2019, allocate, aggregate

Pull every 2010 tract for 2019 in each county that appears in the weights table (usually just
yours), build the 11-digit code, and merge with the weights. Then multiply every estimate and
every MOE by the weight, and aggregate as in Lab 4: sum the estimates, root sum of squares for
the MOEs. `pull_geo` and `aggregate_tracts` are unchanged; the only new step is the multiply.

In [ ]:
def crosswalk_2019(weights, variables, name):
    '''Allocate 2015 to 2019 tract estimates (2010 tracts) onto a neighborhood defined by 2020 tracts.

    Inputs:
    - weights: DataFrame with GEOID_TRACT_10, weight (total share of that 2010 tract inside the neighborhood), county_10
    - variables: a variables dictionary (any count table)
    - name: the neighborhood name for the NAME column

    Output: a one-row DataFrame, the neighborhood in 2015 to 2019, with MOEs by root sum of squares.
    '''
    frames = []
    for county_code in sorted(weights['county_10'].unique()):
        st, co = county_code[:2], county_code[2:]
        df = pull_geo('tract:*', f'state:{st} county:{co}', variables, year=2019)
        frames.append(df)
    all_2019 = pd.concat(frames, ignore_index=True)
    all_2019['GEOID_TRACT_10'] = all_2019['state'] + all_2019['county'] + all_2019['tract']

    merged = weights.merge(all_2019, on='GEOID_TRACT_10', how='left')
    not_found = merged[merged['NAME'].isna()]['GEOID_TRACT_10'].tolist()
    if not_found:
        print('No 2019 data returned for these 2010 tracts (check the codes):', not_found)
        merged = merged[merged['NAME'].notna()].copy()

    data_cols = [col for col in variables.values() if col not in ('NAME', 'GEO_ID')]
    merged[data_cols] = merged[data_cols].mul(merged['weight'], axis='rows')   # estimate x weight, MOE x weight

    out = aggregate_tracts(merged[data_cols], name)
    out.insert(2, 'total_weight', merged['weight'].sum())
    return out, merged

nbhd_2019, allocation = crosswalk_2019(weights, variables_of_interest, f'{NEIGHBORHOOD_NAME}, 2015 to 2019')
allocation[['GEOID_TRACT_10', 'weight', 'total', 'total_moe', 'hispanic', 'hispanic_moe']].round(1)

`n_tracts` counts the 2010 tracts that contributed; `total_weight` is the sum of their weights,
which is the neighborhood's size in "2010 tracts' worth of land." For the example it is close
to 2 (one whole tract 4229, most of tract 4226, and slivers). `.mul(..., axis='rows')`
multiplies every data column by the one weight column, row by row.

In [ ]:
nbhd_2019[['NAME', 'n_tracts', 'total_weight', 'total', 'total_moe', 'hispanic', 'hispanic_moe', 'nh_asian', 'nh_asian_moe']].round(1)

### 5.1 What the direct pull would have given you

For comparison, ask the 2019 API for the 2020 tract numbers directly, the way Lab 5 Section 6.2
does. Watch how many come back.

In [ ]:
direct = pull_geo('tract:*', f'state:{STATE} county:{COUNTY}', variables_of_interest, year=2019)
direct = direct[direct['tract'].isin(TRACT_LIST)]
print(f'Direct 2019 pull of the 2020 numbers: {len(direct)} of {len(TRACT_LIST)} tracts found.')

Zero of three, with no error message. A renumbered tract and a split tract simply do not
exist in the 2019 table under their 2020 names. Had a boundary merely shifted, the direct pull
would have returned the old shape under the same number, which is worse: a number that looks
right and describes different land.

## 6. The 2024 row, the shares, and the test

The 2024 row comes straight from the Lab 4 chain. Then the Lab 5 test, with the periods as
row labels.

In [ ]:
all_2024 = pull_geo('tract:*', f'state:{STATE} county:{COUNTY}', variables_of_interest, year=2024)
tracts_2024 = all_2024[all_2024['tract'].isin(TRACT_LIST)].copy()
print(f'2024: {len(tracts_2024)} of {len(TRACT_LIST)} tracts found.')
nbhd_2024 = aggregate_tracts(tracts_2024, f'{NEIGHBORHOOD_NAME}, 2020 to 2024')

both = pd.concat([nbhd_2019, nbhd_2024], ignore_index=True)
both['period'] = ['2015 to 2019', '2020 to 2024']
both = add_standard_errors(add_shares(both.set_index('period'), GROUPS))

for g in GROUPS:
    z_g = z_statistic(both, f'pct_{g}', '2015 to 2019', '2020 to 2024')
    print(f'{g:10} {both.loc["2015 to 2019", f"pct_{g}"]:6.1f}%  ->  {both.loc["2020 to 2024", f"pct_{g}"]:6.1f}%   Z = {z_g:5.2f}   {significance(z_g)}')

both[['NAME', 'n_tracts', 'total', 'total_moe', 'pct_hispanic', 'pct_hispanic_moe', 'pct_nh_asian', 'pct_nh_asian_moe']].round(1)

In [ ]:
# EXERCISE C1: set Section 2 to your own 2020 tracts and rerun Sections 4 to 6. Compare the
# crosswalked 2019 total with what Lab 5 Section 6.2 gave you from a 2010 list, if you have one.
# In a comment: which 2010 tracts contributed less than their whole selves, and how much?

In [ ]:
# EXERCISE C2: write the Data Note for a Part II exhibit built this way, as a comment. It should
# name the relationship file, the weighting (share of 2010 land area), the number of 2010 tracts
# that contributed, and the assumption the method makes.

## 7. Writing it up, and the better tools

**Data Note wording.** Something like: "2015 to 2019 values are allocated from 2010 census
tracts to the 2020 tracts in the neighborhood using the Census Bureau's 2020 tract relationship
file for California, weighting each 2010 tract by the share of its land area inside the
neighborhood (N 2010 tracts contributed). The method assumes residents are evenly distributed
across each tract's land area. Margins of error are scaled by the same weights and combined by
root sum of squares."

**Citing the file (APA 7):**

U.S. Census Bureau. (2021). *2020 Census tract to 2010 Census tract relationship file: California*
(tab20_tract20_tract10_st06.txt) [Data set].
https://www.census.gov/geographies/reference-files/time-series/geo/relationship-files.2020.html

**Better weights.** Land area is the crudest weight. Two free tools weight by where people
lived, using block-level counts, which is much better when a boundary cuts across a
populated area:

* **IPUMS NHGIS geographic crosswalks** (https://www.nhgis.org/geographic-crosswalks): a
  2010-tract-to-2020-tract crosswalk with weights based on population, housing units, and other
  block-level counts; free with registration. Their guidance says to start from block groups
  rather than tracts when the table is published for block groups. If you use one, the code in
  Section 5 is the same with their weight column in place of `w10`.
* **The Longitudinal Tract Data Base** (Logan, Xu, & Stults, 2014;
  https://s4.ad.brown.edu/projects/diversity/researcher/bridging.htm): harmonizes data to
  **2010** boundaries, the opposite direction from this notebook, with a 2020-block-to-2010-tract
  crosswalk. Useful if you would rather hold the 2010 boundaries fixed and bring 2024 back to
  them.

Logan, J. R., Xu, Z., & Stults, B. J. (2014). Interpolating U.S. decennial census tract data
from as early as 1970 to 2010: A longitudinal tract database. *The Professional Geographer,
66*(3), 412–420. https://doi.org/10.1080/00330124.2014.905156

**What does not crosswalk.** A median cannot be multiplied by a weight and summed; if your Part
II question is about median income or rent at the neighborhood scale, crosswalk the
distribution table instead (B19001 for income, B25063 for rent, both counts), then re-derive
the median from the bins. A mean can be rebuilt from a published aggregate (B19025 aggregate
household income, B08013 aggregate travel time) crosswalked as a count and divided by the
crosswalked household or worker count. Both are advanced-path work; ask in office hours before
you commit to one.